# Crowd Crush & Surge Detection Video Pipeline
**Meenakshi Gopakumar Nair — Undergraduate Project**

This notebook:
1. Installs a version of Ultralytics that fixes the PyTorch 2.6 loading issue
2. Downloads the VisDrone dataset and fine tunes YOLOv8m on it
3. Runs the full crowd detection pipeline on drone video footage, either from the VisDrone dataset or other

---
**Before running**: Set runtime to **L4 GPU** via Runtime → Change runtime type or else it will really take too long

## Install Dependencies

We're use `ultralytics>=8.3.70` which added official PyTorch 2.6 support. This fixes the errors from earlier versions that happened with older versions.

In [ ]:
!pip install "ultralytics>=8.3.70" --quiet
!pip install opencv-python-headless matplotlib numpy Pillow PyYAML --quiet

# Confirm versions
import torch
import ultralytics
print(f" PyTorch version  : {torch.__version__}")
print(f" Ultralytics version: {ultralytics.__version__}")
print(f" GPU available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name         : {torch.cuda.get_device_name(0)}")

## Mount Google Drive

Make sure the VisDrone valset zip should is in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')
import os
print('Files in Drive root:')
print(os.listdir('/content/drive/MyDrive'))

## VisDrone Dataset & Prepare for Training

We need:
- **Valset** for testing
- **Training set** for fine-tuning the model
- [**Find Them Here**](https://https://github.com/VisDrone/VisDrone-Dataset)

This cell downloads it into Colab so you don't need to go through Drive.

In [ ]:
import os

# ── Unzip valset from Drive (for testing later) ──────────────────────────────
VALSET_ZIP = '/content/drive/MyDrive/VisDrone2019-DET-val.zip'

if os.path.exists(VALSET_ZIP):
    print('Unzipping valset...')
    !unzip -q "$VALSET_ZIP" -d /content/visdrone_val
    print(' Valset ready at /content/visdrone_val')
else:
    print(f'  Could not find {VALSET_ZIP}')
    print('Check your Drive filename matches exactly.')

In [ ]:
# ── Download training set directly into Colab ────────────────────────────────
# This is 1.44 GB and takes a few minutes — run once per session
import os

TRAINSET_PATH = '/content/VisDrone2019-DET-train.zip'

if not os.path.exists(TRAINSET_PATH):
    print('Downloading VisDrone training set (1.44 GB)...')
    !wget -q --show-progress \
        "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip" \
        -O /content/VisDrone2019-DET-train.zip
    print(' Download complete.')
else:
    print('Training zip already exists, skipping download.')

print('Unzipping training set...')
!unzip -q /content/VisDrone2019-DET-train.zip -d /content/visdrone_train
print(' Training set ready.')

## Convert VisDrone Annotations to YOLO Format

VisDrone annotations are stored in a custom format, one `.txt` file per image with columns:
`x, y, w, h, score, category, truncation, occlusion`

YOLO expects:
`class_id  cx_norm  cy_norm  w_norm  h_norm`

So run this cell  to convert them

In [ ]:
import os
import glob
from pathlib import Path

# VisDrone class IDs we care about for crowd detection
# 1=pedestrian, 2=people (VisDrone is 1-indexed, YOLO is 0-indexed)
KEEP_CLASSES = {1: 0, 2: 1}  # VisDrone_id -> YOLO_id

def convert_visdrone_to_yolo(src_img_dir, src_ann_dir, dst_img_dir, dst_lbl_dir):
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    ann_files = glob.glob(os.path.join(src_ann_dir, '*.txt'))
    converted = 0
    skipped   = 0

    for ann_path in ann_files:
        stem     = Path(ann_path).stem
        img_path = os.path.join(src_img_dir, stem + '.jpg')

        if not os.path.exists(img_path):
            skipped += 1
            continue

        # Read image dimensions
        import cv2
        img = cv2.imread(img_path)
        if img is None:
            skipped += 1
            continue
        H, W = img.shape[:2]

        yolo_lines = []
        with open(ann_path) as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) < 6:
                    continue
                x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                cat = int(parts[5])

                if cat not in KEEP_CLASSES:
                    continue
                if w <= 0 or h <= 0:
                    continue

                # Convert to YOLO normalised centre format
                cx = (x + w / 2) / W
                cy = (y + h / 2) / H
                nw = w / W
                nh = h / H

                yolo_id = KEEP_CLASSES[cat]
                yolo_lines.append(f"{yolo_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        # Copy image and write label
        import shutil
        shutil.copy(img_path, os.path.join(dst_img_dir, stem + '.jpg'))
        with open(os.path.join(dst_lbl_dir, stem + '.txt'), 'w') as f:
            f.write('\n'.join(yolo_lines))
        converted += 1

    print(f'  Converted: {converted} | Skipped: {skipped}')

print('Converting training set...')
convert_visdrone_to_yolo(
    '/content/visdrone_train/VisDrone2019-DET-train/images',
    '/content/visdrone_train/VisDrone2019-DET-train/annotations',
    '/content/dataset/images/train',
    '/content/dataset/labels/train'
)

print('Converting validation set...')
convert_visdrone_to_yolo(
    '/content/visdrone_val/VisDrone2019-DET-val/images',
    '/content/visdrone_val/VisDrone2019-DET-val/annotations',
    '/content/dataset/images/val',
    '/content/dataset/labels/val'
)

print(' Dataset conversion complete.')

In [ ]:
# Write the YOLO dataset config file
import yaml

dataset_config = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val':   'images/val',
    'names': {0: 'pedestrian', 1: 'people'}
}

with open('/content/visdrone_crowd.yaml', 'w') as f:
    yaml.dump(dataset_config, f)

print(' Dataset config written to /content/visdrone_crowd.yaml')
print(yaml.dump(dataset_config))

## Fine Tuning YOLOv8m on VisDrone

This trains YOLOv8 medium on the VisDrone dataset so it learns to detect people from drone altitude.

**Estimated time on L4 GPU:**  Around 45–60 minutes for 30 epochs

This only needs to be run **once**. After training the weights are saved and you can skip this cell in the future.

In [ ]:
from ultralytics import YOLO
import os

TRAINED_WEIGHTS = '/content/runs/crowd_detect/weights/best.pt'

if os.path.exists(TRAINED_WEIGHTS):
    print(f' Trained weights already exist at {TRAINED_WEIGHTS}')
    print('Skipping training — delete the file to retrain.')
else:
    print('Starting training...')
    print('This will take ~45–60 minutes on L4 GPU.')

    model = YOLO('yolov8m.pt')  # start from COCO pretrained weights

    results = model.train(
        data    = '/content/visdrone_crowd.yaml',
        epochs  = 30,
        imgsz   = 640,
        batch   = 16,         # fits comfortably on L4
        device  = 0,          # GPU
        project = '/content/runs',
        name    = 'crowd_detect',
        patience= 10,         # stop early if no improvement
        workers = 4,
        verbose = True
    )

    print(f'\n Training complete!')
    print(f'Best weights saved at: {TRAINED_WEIGHTS}')

## Load Trained Model

After training we're loading the best checkpoint. This also runs a quick test on a single valset image to confirm everything is working.

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os, math, glob

TRAINED_WEIGHTS = '/content/runs/crowd_detect/weights/best.pt'

print('Loading trained model...')
model = YOLO(TRAINED_WEIGHTS)
model.overrides['conf']    = 0.25
model.overrides['iou']     = 0.45
model.overrides['max_det'] = 1000

PERSON_CLASSES = [0, 1]  # pedestrian, people

print(' Model loaded.')
print('Classes:', model.model.names)

# Quick test on one val image
val_images = glob.glob('/content/dataset/images/val/*.jpg')
if val_images:
    test_img = val_images[0]
    results  = model(test_img, verbose=False)
    boxes    = results[0].boxes.xyxy.cpu().numpy()
    # Filter to people only
    cls      = results[0].boxes.cls.cpu().numpy().astype(int)
    boxes    = boxes[np.isin(cls, PERSON_CLASSES)]

    annotated = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 7))
    plt.imshow(annotated)
    plt.title(f'Test detection on val image — {len(boxes)} people found', fontsize=13)
    plt.axis('off')
    plt.show()
    print(f' Test passed — detected {len(boxes)} people.')

## Pipeline Configuration

In [ ]:
# ── Grid settings ──────────────────────────────────────────────────────────
GRID_ROWS = 4
GRID_COLS = 4

# ── Drone camera assumptions ────────────────────────────────────────────────
DRONE_ALTITUDE_M = 30
CAMERA_FOV_DEG   = 84

# ── Processing ──────────────────────────────────────────────────────────────
FRAME_SKIP = 2   # process every Nth frame (1=every frame, 2=every other)

# ── Risk level thresholds (people/m²) ───────────────────────────────────────
RISK_LEVELS = [
    (7.0, 'CRITICAL', (200, 50,  50)),
    (5.0, 'DANGER',   (210, 110, 50)),
    (4.0, 'WARNING',  (210, 185, 50)),
    (2.0, 'CAUTION',  (140, 190, 80)),
    (0.0, 'SAFE',     (50,  170, 80)),
]
RISK_ORDER   = ['SAFE', 'CAUTION', 'WARNING', 'DANGER', 'CRITICAL']
RISK_COLOURS = {l: c for _, l, c in RISK_LEVELS}

def density_to_risk(density):
    for threshold, label, colour in RISK_LEVELS:
        if density >= threshold:
            return label, colour
    return 'SAFE', (50, 170, 80)

def bump_risk(label, motion_label):
    if motion_label in ('HIGH CHAOS', 'MODERATE CHAOS'):
        idx = RISK_ORDER.index(label)
        return RISK_ORDER[min(idx + 1, len(RISK_ORDER) - 1)]
    return label

def motion_risk_label(disorder, avg_speed):
    if disorder > 1.8 and avg_speed > 3.0:
        return 'HIGH CHAOS',     (200, 50,  50)
    elif disorder > 1.4 or avg_speed > 2.0:
        return 'MODERATE CHAOS', (210, 110, 50)
    elif disorder > 1.0:
        return 'LOW MOTION',     (210, 185, 50)
    else:
        return 'CALM',           (50,  170, 80)

print(' Parameters configured.')

## Pipeline Functions

In [ ]:
def detect_people(frame, model):
    """Detect people in a frame, return bounding boxes."""
    cv2.imwrite('_tmp.jpg', frame)
    results  = model('_tmp.jpg', verbose=False)
    all_boxes = results[0].boxes.xyxy.cpu().numpy()
    all_cls   = results[0].boxes.cls.cpu().numpy().astype(int)
    mask      = np.isin(all_cls, PERSON_CLASSES)
    return all_boxes[mask]


def compute_density_grid(boxes, image_shape):
    """Count people per grid cell, convert to people/m²."""
    h, w = image_shape[:2]
    ground_width  = 2 * math.tan(math.radians(CAMERA_FOV_DEG / 2)) * DRONE_ALTITUDE_M
    ground_height = ground_width * (h / w)
    cell_area     = (ground_width / GRID_COLS) * (ground_height / GRID_ROWS)

    count_grid = np.zeros((GRID_ROWS, GRID_COLS), dtype=int)
    for box in boxes:
        cx  = (box[0] + box[2]) / 2
        cy  = (box[1] + box[3]) / 2
        col = min(int(cx / w * GRID_COLS), GRID_COLS - 1)
        row = min(int(cy / h * GRID_ROWS), GRID_ROWS - 1)
        count_grid[row][col] += 1

    density = count_grid / max(cell_area, 0.001)
    return density, count_grid


def compute_optical_flow(frame1, frame2):
    """Measure how chaotically the crowd is moving between frames."""
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    flow  = cv2.calcOpticalFlowFarneback(gray1, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    return float(np.std(angle)), float(np.mean(mag))


def smooth_density(prev, current, alpha=0.4):
    """Smooth density values between frames to reduce flickering."""
    return current if prev is None else alpha * current + (1 - alpha) * prev


def generate_risk_map(density_grid, motion_label):
    """Combine density + motion into a final risk level per zone."""
    risk_map = []
    alerts   = []
    for r in range(GRID_ROWS):
        row = []
        for c in range(GRID_COLS):
            base, _     = density_to_risk(density_grid[r][c])
            final       = bump_risk(base, motion_label)
            row.append(final)
            if RISK_ORDER.index(final) >= RISK_ORDER.index('WARNING'):
                alerts.append({
                    'zone':       f'R{r+1}C{c+1}',
                    'density':    round(density_grid[r][c], 2),
                    'final_risk': final,
                })
        risk_map.append(row)
    return risk_map, alerts


print('Pipeline functions defined.')

## Frame Annotation Function

In [ ]:
def annotate_frame(frame, boxes, density_grid, risk_map, alerts,
                   motion_label, motion_colour, disorder, avg_speed,
                   frame_idx, total_people):
    """Draw the full risk overlay onto a single frame."""
    out    = frame.copy()
    h, w   = out.shape[:2]
    cell_h = h // GRID_ROWS
    cell_w = w // GRID_COLS

    # Draw bounding boxes around detected people
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (100, 200, 255), 1)

    # Draw semi-transparent risk grid
    overlay = out.copy()
    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            risk   = risk_map[r][c]
            colour = RISK_COLOURS[risk]
            x1, y1 = c * cell_w, r * cell_h
            x2, y2 = x1 + cell_w, y1 + cell_h
            cv2.rectangle(overlay, (x1, y1), (x2, y2), colour, -1)
            cv2.rectangle(out, (x1, y1), (x2, y2), colour, 2)
            cv2.putText(out, risk,
                        (x1 + 4, y1 + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.38, colour, 1)
            cv2.putText(out, f'{density_grid[r][c]:.2f}/m2',
                        (x1 + 4, y1 + 34), cv2.FONT_HERSHEY_SIMPLEX, 0.32, colour, 1)
    out = cv2.addWeighted(overlay, 0.18, out, 0.82, 0)

    # Top HUD bar
    cv2.rectangle(out, (0, 0), (w, 55), (15, 15, 30), -1)
    cv2.putText(out, f'CROWD CRUSH DETECTION  |  Frame: {frame_idx}',
                (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 255), 1)
    cv2.putText(out,
                f'People: {total_people}   Motion: {motion_label}   '
                f'Disorder: {disorder:.2f}   Alerts: {len(alerts)}',
                (8, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.46, motion_colour, 1)

    # Alert panel bottom-right
    if alerts:
        px = w - 230
        py = h - (len(alerts) * 22 + 32)
        cv2.rectangle(out, (px - 5, py - 22), (w - 5, h - 5), (15, 15, 30), -1)
        cv2.putText(out, f'{len(alerts)} ALERT(S)',
                    (px, py), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 80, 80), 1)
        for i, a in enumerate(alerts[:6]):
            col = RISK_COLOURS.get(a['final_risk'], (255, 255, 255))
            cv2.putText(out, f"{a['zone']}: {a['final_risk']} ({a['density']}/m2)",
                        (px, py + 20 + i * 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.34, col, 1)

    return out


print(' Annotation function defined.')

## Set Your Video File

Point this at a video file in your Google Drive.
The VisDrone VID valset works for this. Download it from the VisDrone GitHub
(Object Detection in Videos → valset → GoogleDrive) and upload to Drive.

Alternatively any drone footage of crowds would still work.

In [ ]:
# ── SET YOUR VIDEO PATH HERE ───────────────────────────────────────────────
VIDEO_PATH = '/content/drive/MyDrive/YOUR_VIDEO.mp4'

# If you have the VisDrone VID valset:
#!unzip -q "/content/drive/MyDrive/VisDrone2019-VID-val.zip" -d /content/visdrone_vid
#import glob
#videos = glob.glob('/content/visdrone_vid/**/*.mp4', recursive=True)
#if not videos:
    #videos = glob.glob('/content/visdrone_vid/**/*.avi', recursive=True)
#print(videos)
VIDEO_PATH = videos[0]

if os.path.exists(VIDEO_PATH):
    cap   = cv2.VideoCapture(VIDEO_PATH)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f' Video: {VIDEO_PATH}')
    print(f'   {w}x{h_} @ {fps:.1f}fps — {total} frames ({total/fps:.1f}s)')
else:
    print(f' File not found: {VIDEO_PATH}')
    print('Update VIDEO_PATH above to your actual file.')

## Run the Full Video Pipeline

Processes every Nth frame (set by `FRAME_SKIP`), runs all pipeline stages, and writes an annotated output video.

In [ ]:
OUTPUT_VIDEO = '/content/crowd_analysis_output.mp4'

cap    = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS) or 25
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps / FRAME_SKIP,
    (width, height)
)

prev_frame       = None
prev_density     = None
frame_idx        = 0
processed        = 0
all_people        = []
all_densities     = []
all_alert_counts  = []

print(f'Processing video — every {FRAME_SKIP} frames...')
print(f'Estimated frames to process: {total // FRAME_SKIP}')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP != 0:
        frame_idx += 1
        continue

    # Detection
    boxes = detect_people(frame, model)

    # Density
    d_grid, _ = compute_density_grid(boxes, frame.shape)
    d_grid    = smooth_density(prev_density, d_grid)
    prev_density = d_grid.copy()

    # Optical flow
    if prev_frame is not None:
        disorder, avg_speed = compute_optical_flow(prev_frame, frame)
    else:
        disorder, avg_speed = 0.0, 0.0

    m_label, m_colour = motion_risk_label(disorder, avg_speed)

    # Risk
    risk_map, alerts = generate_risk_map(d_grid, m_label)

    # Annotate and write
    annotated = annotate_frame(
        frame, boxes, d_grid, risk_map, alerts,
        m_label, m_colour, disorder, avg_speed,
        frame_idx, len(boxes)
    )
    out_writer.write(annotated)

    # Stats
    all_people.append(len(boxes))
    all_densities.append(d_grid.max())
    all_alert_counts.append(len(alerts))

    prev_frame = frame.copy()
    processed += 1
    frame_idx += 1

    if processed % 25 == 0:
        print(f'  Frame {processed} | People: {len(boxes)} | '
              f'Max density: {d_grid.max():.2f}/m² | Alerts: {len(alerts)}')

cap.release()
out_writer.release()
print(f'\n Done! Processed {processed} frames.')
print(f' Output saved to: {OUTPUT_VIDEO}')

## Session Statistics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#1a1a2e')

frames_x = list(range(len(all_people)))

axes[0].plot(frames_x, all_people, color='#64b5f6', linewidth=1.5)
axes[0].fill_between(frames_x, all_people, alpha=0.3, color='#64b5f6')
axes[0].set_title('People Detected Per Frame', color='white', fontsize=11)
axes[0].set_facecolor('#0f0f23')
axes[0].tick_params(colors='white')

axes[1].plot(frames_x, all_densities, color='#ef5350', linewidth=1.5)
axes[1].fill_between(frames_x, all_densities, alpha=0.3, color='#ef5350')
axes[1].axhline(y=4.0, color='#ffcc00', linestyle='--', linewidth=1, label='Warning (4/m²)')
axes[1].axhline(y=5.0, color='#ff6600', linestyle='--', linewidth=1, label='Danger (5/m²)')
axes[1].axhline(y=7.0, color='#ff0000', linestyle='--', linewidth=1, label='Critical (7/m²)')
axes[1].legend(fontsize=7, facecolor='#1a1a2e', labelcolor='white')
axes[1].set_title('Max Zone Density Over Time', color='white', fontsize=11)
axes[1].set_facecolor('#0f0f23')
axes[1].tick_params(colors='white')

axes[2].bar(frames_x, all_alert_counts, color='#ff7043', width=1.0)
axes[2].set_title('Zones Flagged Per Frame', color='white', fontsize=11)
axes[2].set_facecolor('#0f0f23')
axes[2].tick_params(colors='white')

plt.suptitle('Video Session Statistics', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/session_stats.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print(f'Peak people detected : {max(all_people) if all_people else 0}')
print(f'Peak density         : {max(all_densities):.2f} people/m²' if all_densities else 'N/A')
print(f'Alert frames         : {sum(1 for a in all_alert_counts if a > 0)}')

## Download Outputs

In [ ]:
from google.colab import files

print('Downloading annotated video...')
files.download(OUTPUT_VIDEO)

print('Downloading stats chart...')
files.download('/content/session_stats.png')

print('\nTo save trained weights to Drive for future sessions:')
print('Run the cell below ↓')

In [ ]:
import shutil

# Save trained weights to Drive so you don't need to retrain next session
DRIVE_WEIGHTS = '/content/drive/MyDrive/visdrone_crowd_yolov8m_best.pt'
shutil.copy('/content/runs/crowd_detect/weights/best.pt', DRIVE_WEIGHTS)
print(f'Weights saved to Drive: {DRIVE_WEIGHTS}')
print('Next session: load with YOLO(DRIVE_WEIGHTS) and skip the training step.')


**Next steps:**
1. Wrap pipeline in a ROS2 node for drone integration
2. Feed VisDrone VID footage through ROS2 as the camera topic
3. Replace grid density with CSRNet for smoother heatmaps